# DELBERT Inference Example

This notebook demonstrates how to load a DELBERT model from HuggingFace Hub and run predictions on molecular fingerprints.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Ensure repo root is on the path (run notebook from DELBERT/ root)
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

from inference.predict import predict, predict_from_parquet, load_model

## 2. Load the model

The model is downloaded from HuggingFace Hub on first use and cached locally.

In [2]:
MODEL_ID = "wanglab/delbert-wdr91"

model, tokenizer, device = load_model(MODEL_ID, from_hub=True)
print(f"Model loaded on: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size}")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Model loaded on: cpu
Parameters: 70,882,562
Vocab size: 14458


## 3. Predict from a single molecule

Provide a dict with 4 dense fingerprint arrays (length 2048 each):
- `ECFP4`, `FCFP6`, `ATOMPAIR`, `TOPTOR`

These are the same arrays as in AIRCHECK parquet files.

In [3]:
import pandas as pd
import numpy as np

# Load example molecules (10 molecules: 5 active + 5 inactive)
# For full datasets, download from https://aircheck.ai/datasets
df = pd.read_parquet(project_root / "data/WDR91_10-examples.parquet")
print(f"Loaded {len(df)} molecules")
df[["COMPOUND_ID", "LABEL"]]

Loaded 10 molecules


,COMPOUND_ID,LABEL
0,49018705640097,1
1,45005301630560,0
2,49004305360154,1
3,4001100461194,1
4,36000801361300,0
5,4000303801418,1
6,44014800261096,0
7,30041002340431,0
8,49004304190154,1
9,23010100090265,0


In [4]:
# Pick one molecule and build a dict with its fingerprints
row = df.iloc[0]
molecule = {
    "ECFP4": row["ECFP4"],
    "FCFP6": row["FCFP6"],
    "ATOMPAIR": row["ATOMPAIR"],
    "TOPTOR": row["TOPTOR"],
}

# Each FP is a dense int32 array of length 2048
for name, arr in molecule.items():
    arr = np.asarray(arr)
    print(f"{name}: shape={arr.shape}, nonzero={np.count_nonzero(arr)}, max={arr.max()}")

ECFP4: shape=(2048,), nonzero=59, max=5
FCFP6: shape=(2048,), nonzero=67, max=13
ATOMPAIR: shape=(2048,), nonzero=233, max=8
TOPTOR: shape=(2048,), nonzero=29, max=7


In [5]:
# Predict
probs = predict(molecule, model_path=MODEL_ID)

print(f"Compound: {row['COMPOUND_ID']}")
print(f"True label: {row['LABEL']}")
print(f"Predicted P(active): {probs[0]:.4f}")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Compound: 49018705640097
True label: 1
Predicted P(active): 0.7695


## 4. Predict a batch of molecules

In [6]:
# Build a list of molecule dicts from the dataframe
molecules = []
for i in range(len(df)):
    r = df.iloc[i]
    molecules.append({fp: r[fp] for fp in ["ECFP4", "FCFP6", "ATOMPAIR", "TOPTOR"]})

probs = predict(molecules, model_path=MODEL_ID)

for i, p in enumerate(probs):
    label_str = "active" if df.iloc[i]["LABEL"] == 1 else "inactive"
    print(f"  {df.iloc[i]['COMPOUND_ID']}: P(active)={p:.4f}  ({label_str})")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

  49018705640097: P(active)=0.7695  (active)
  45005301630560: P(active)=0.7744  (inactive)
  49004305360154: P(active)=0.9738  (active)
  4001100461194: P(active)=0.6512  (active)
  36000801361300: P(active)=0.0210  (inactive)
  4000303801418: P(active)=0.1330  (active)
  44014800261096: P(active)=0.2431  (inactive)
  30041002340431: P(active)=0.1394  (inactive)
  49004304190154: P(active)=0.8238  (active)
  23010100090265: P(active)=0.1921  (inactive)


## 5. Predict from a parquet file

For large-scale inference, pass a parquet file path directly. The parquet must have columns: `ECFP4`, `FCFP6`, `ATOMPAIR`, `TOPTOR` (and optionally `COMPOUND_ID`).

Full datasets can be downloaded from [AIRCHECK](https://aircheck.ai/datasets).

In [7]:
result = predict_from_parquet(
    str(project_root / "data/WDR91_10-examples.parquet"),
    model_path=MODEL_ID,
    batch_size=200,
)

print(f"Predictions: {len(result)}")
result

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Preparing 10 molecules...
Running inference (batch_size=200)...
Done. 10 predictions.
Predictions: 10


,compound_id,probability
0,49018705640097,0.769472
1,45005301630560,0.774409
2,49004305360154,0.973784
3,4001100461194,0.651165
4,36000801361300,0.021043
5,4000303801418,0.133039
6,44014800261096,0.243077
7,30041002340431,0.139423
8,49004304190154,0.823781
9,23010100090265,0.192107


## 6. Step-by-step: what happens under the hood

For those who want to understand or customize the inference pipeline.

In [8]:
import torch
from inference.predict import (
    dense_to_sparse,
    dense_dict_to_sparse_row,
    prepare_molecule,
    FINGERPRINT_TYPES,
    TOKEN_FORMAT,
)
from delbert.data.transforms import molecule_to_tokens, MolecularCollator

# Pick a molecule
row = df.iloc[0]
molecule = {fp: row[fp] for fp in FINGERPRINT_TYPES}

In [9]:
# Step 1: Dense array -> sparse (indices, values)
indices, values = dense_to_sparse(molecule["FCFP6"])
print(f"FCFP6 sparse: {len(indices)} nonzero bits")
print(f"  First 5 indices: {indices[:5]}")
print(f"  First 5 values:  {values[:5]}")

FCFP6 sparse: 67 nonzero bits
  First 5 indices: [0, 1, 2, 4, 6]
  First 5 values:  [13, 1, 3, 8, 4]


In [10]:
# Step 2: Convert all FPs to sparse row format
sparse_row = dense_dict_to_sparse_row(molecule)
print(f"Sparse row keys: {list(sparse_row.keys())}")
print(f"ECFP4: {len(sparse_row['ECFP4_indices'])} bits set")
print(f"FCFP6: {len(sparse_row['FCFP6_indices'])} bits set")

Sparse row keys: ['ECFP4_indices', 'ECFP4_values', 'FCFP6_indices', 'FCFP6_values', 'ATOMPAIR_indices', 'ATOMPAIR_values', 'TOPTOR_indices', 'TOPTOR_values']
ECFP4: 59 bits set
FCFP6: 67 bits set


In [11]:
# Step 3: Convert to token strings
tokens, segment_ids = molecule_to_tokens(
    sparse_row,
    FINGERPRINT_TYPES,
    return_segment_ids=True,
    token_format=TOKEN_FORMAT,
)
print(f"Total tokens: {len(tokens)}")
print(f"First 5 tokens: {tokens[:5]}")
print(f"First 5 segment IDs: {segment_ids[:5]}")
print(f"\nSegment ID mapping: ECFP4=1, FCFP6=2, ATOMPAIR=3, TOPTOR=4")
for seg_id, fp_name in enumerate(FINGERPRINT_TYPES, 1):
    count = segment_ids.count(seg_id)
    print(f"  {fp_name} (seg={seg_id}): {count} tokens")

Total tokens: 388
First 5 tokens: ['ECFP4_1_1', 'ECFP4_32_1', 'ECFP4_80_2', 'ECFP4_124_1', 'ECFP4_142_1']
First 5 segment IDs: [1, 1, 1, 1, 1]

Segment ID mapping: ECFP4=1, FCFP6=2, ATOMPAIR=3, TOPTOR=4
  ECFP4 (seg=1): 59 tokens
  FCFP6 (seg=2): 67 tokens
  ATOMPAIR (seg=3): 233 tokens
  TOPTOR (seg=4): 29 tokens


In [12]:
# Step 4: Map tokens to IDs via vocabulary
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Input IDs (first 5): {input_ids[:5]}")
print(f"Sequence length: {len(input_ids)}")

# Check for unknown tokens
unk_id = tokenizer.convert_tokens_to_ids("<unk>")
n_unk = input_ids.count(unk_id)
print(f"Unknown tokens: {n_unk} / {len(input_ids)}")

Input IDs (first 5): [8229, 8491, 9193, 7092, 7356]
Sequence length: 388
Unknown tokens: 0 / 388


In [13]:
# Step 5: Pad and batch (MolecularCollator handles this)
collator = MolecularCollator(pad_token_id=tokenizer.pad_token_id)
batch = collator([{"input_ids": input_ids, "segment_ids": segment_ids}])

print(f"Batch tensors:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

Batch tensors:
  input_ids: torch.Size([1, 388])
  attention_mask: torch.Size([1, 388])
  segment_ids: torch.Size([1, 388])


In [14]:
# Step 6: Forward pass
with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        segment_ids=batch["segment_ids"].to(device),
    )

logits = outputs["logits"]
probs = torch.softmax(logits, dim=-1)
print(f"Logits: {logits.cpu().numpy().flatten()}")
print(f"P(inactive): {probs[0, 0]:.4f}")
print(f"P(active):   {probs[0, 1]:.4f}")

Logits: [-0.6590785  0.5462567]
P(inactive): 0.2305
P(active):   0.7695
